In [ ]:
# Import Python and ViT modules

import os, time
import random
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import transforms, datasets
import torch.optim as optim
from PIL import Image
import torch.nn.functional as F
from torch.optim.lr_scheduler import OneCycleLR



import geoclip
from geoclip import GeoCLIP

In [ ]:
# Look at the data distribution

file_list = []
image_count = []

for f in os.listdir():
    try:
        file_list.append(int(f))
    except:
        continue
    image_count.append(len(os.listdir(f)))

print(f'There are {len(file_list)} quadrants')

In [ ]:
plt.bar(file_list, image_count)
plt.xlabel('Quad')
plt.ylabel('Amount of images per quad')

In [ ]:
# Display 5 random images

random_quads = random.sample(file_list, 5)

fig, axs = plt.subplots(1, 5, figsize=(20,4))


for quad, ax in zip(random_quads, axs):
    try:
        rand_img = random.choice(os.listdir(str(quad)))
    except IndexError:
        continue
    img = Image.open(str(quad) + '/' + rand_img)

    ax.imshow(img)
    ax.set_title('IMAGE ID: ' + rand_img.strip('.jpg') + ' QUAD #: ' + str(quad), fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Read in data from train.csv, not including test.csv
# may take a few minutes

world_set = pd.read_csv('train.csv')
us_mask = world_set['unique_country'] == 'US'
us_set = world_set[us_mask]

In [ ]:
# Get a list of all the ids in our folders

all_ids = []

for f in os.listdir():
    try:
        images = os.listdir(f)
    except NotADirectoryError:
        continue
    for im in images:
        try:
            all_ids.append(int(im.strip('.jpg')))
        except ValueError:
            pass



In [ ]:
# Get the dataframe for all the us dataset that we actually use (see get_data.ipynb for more info)

used_mask = us_set['id'].isin(all_ids)
us_set_used = us_set[used_mask]
# A dictionary that maps each id to it's coordinates

id_coords = {idd: (lat, long) for idd, lat, long in zip(us_set_used['id'], us_set_used['latitude'], us_set_used['longitude'])}

In [ ]:
# One metric we will use to compare and predict quadrants in the center, which will be
# the average of all the longitude and latitudes in that quadrant
# centers is the full us_set center and centers_used is the one with only the quadrants we use

def findCenter(lats, longs):
    return (np.mean(lats), np.mean(longs))

centers = {}
quad_list = file_list
centers_used = {}

for quad in us_set['quadtree_10_5000'].unique():
    mask = us_set['quadtree_10_5000'] == quad
    quad_set = us_set[mask]
    lats = np.array(quad_set['latitude'])
    longs = np.array(quad_set['longitude'])
    centers[str(quad)] = findCenter(lats, longs)

for quad in us_set_used['quadtree_10_5000'].unique():
    mask = us_set['quadtree_10_5000'] == quad
    quad_set = us_set[mask]
    lats = np.array(quad_set['latitude'])
    longs = np.array(quad_set['longitude'])
    centers_used[str(quad)] = findCenter(lats, longs)

In [ ]:
# Spot check this visually

mask1 = us_set['quadtree_10_5000'] == 516
print(us_set[mask1]['latitude'])
print(us_set[mask1]['longitude'])

# Pretty close to 20.85, -157 roughly hawaii

In [ ]:
# Simple helper function to get distances between center of cells

def getDistance(centers, quad1, quad2):
    quad1_center = centers[quad1]
    quad2_center = centers[quad2]
    return np.linalg.norm(np.array(quad2_center) - np.array(quad1_center))

getDistance(centers, '516', '554')

In [ ]:
# Visualize the data distribution on a map (increase sample size to see individual cells better)

import folium

m = folium.Map(location=[44, -103], zoom_start=12)
colors = ['blue', 'red', 'cyan', 'yellow', 'orange', 'green', 'white', 'black', 'pink']
rand_rows = us_set_used.sample(1000, random_state=60)

for i in range(len(rand_rows)):
    lat = rand_rows.iloc[i]['latitude']
    long = rand_rows.iloc[i]['longitude']
    folium.CircleMarker(
    location=[lat, long],
    radius=10,                 
    color = colors[rand_rows.iloc[i]['quadtree_10_5000'] % len(colors)],          
    fill=True,
    fill_color="cyan"     
    ).add_to(m)

m